# BTP — Gemma-2-9B-IT proof of concept (step S1 on the main model)

This repeats the M0 checks on **Gemma-2-9B-IT**, in priority order. Stop when compute units run low; every step resumes.

1. **S1b** equivalence checks (about 1 GPU hour)
2. **1,000 GSM8K-train solutions** + parser statistics (about 2–3 hours). This also measures the real speed, so the full S2 cost can be predicted.
3. **SAE validation** on all 42 layers (about 1.5–2 hours)
4. **E0** sparsity check (about 1 hour)

**Before you start (one time):**
* Runtime → Change runtime type → **A100 GPU**. Steps 1–2 also work on an **L4**, which uses fewer units; steps 3–4 need the A100.
* Secrets (key icon, left bar): add `HF_TOKEN` with your Hugging Face *read* token and switch on **Notebook access**.
* Upload `Reasoning-Failure-Diagnosis.zip` to **My Drive / BTP /**.

All outputs go to **My Drive / BTP / results_9b /**. If Colab disconnects: Runtime → Run all again; finished work is skipped.
**When you are done, Runtime → Disconnect and delete runtime**, so units stop being used.

In [ ]:
# 0. GPU and timing log
import subprocess, time, json, pathlib
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True)
assert out.returncode == 0, "No GPU. Runtime -> Change runtime type -> A100 GPU."
GPU, MEM = out.stdout.strip().split(", ")
print(f"GPU: {GPU}, {int(MEM)/1024:.0f} GB")
assert int(MEM) >= 22_000, "Need an A100 (40 GB) or an L4 (24 GB)."
BIG = int(MEM) >= 39_000
print("SAE validation and E0 can run on this GPU" if BIG else "L4: run steps 1-2 only; SAE validation and E0 need the A100")

In [ ]:
# 1. Drive, project, outputs on Drive
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile
BTP = pathlib.Path('/content/drive/MyDrive/BTP')
ZIP = BTP / 'Reasoning-Failure-Diagnosis.zip'
assert ZIP.exists(), f'Upload the project zip to {ZIP}'
RES = BTP / 'results_9b'
for d in ('data_generated', 'reports_m0'):
    (RES / d).mkdir(parents=True, exist_ok=True)
REPO = pathlib.Path('/content/Reasoning-Failure-Diagnosis')
if not REPO.exists():
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content')
for link, target in ((REPO / 'data' / 'generated', RES / 'data_generated'),
                     (REPO / 'reports' / 'm0', RES / 'reports_m0')):
    if not link.is_symlink():
        link.parent.mkdir(parents=True, exist_ok=True)
        if link.exists():
            os.rename(link, str(link) + '.local')
        os.symlink(target, link)
os.chdir(REPO)
LOG = RES / 'timing_log.jsonl'
def log(step, t0, **kw):
    rec = {"step": step, "gpu": GPU, "minutes": round((time.time() - t0) / 60, 1), **kw}
    with open(LOG, 'a') as f: f.write(json.dumps(rec) + '\n')
    print(rec)
print(sorted(os.listdir(REPO)))

In [ ]:
# 2. Python 3.13 environment (see requirements.txt), installed with uv
t0 = time.time()
!pip -q install uv
!uv venv --seed --python 3.13 /content/venv
!uv pip install -q --python /content/venv/bin/python -r requirements.txt
os.environ['VIRTUAL_ENV'] = '/content/venv'
os.environ['PATH'] = '/content/venv/bin:' + os.environ['PATH']
os.environ['PYTHONPATH'] = str(REPO)
!python --version && python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
log('setup', t0)

In [ ]:
# 3. Hugging Face token from the Colab secret (never printed)
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!python -c "from huggingface_hub import whoami; print('HF user:', whoami()['name'])"
!python -c "from huggingface_hub import model_info; model_info('google/gemma-2-9b-it'); print('Gemma-2-9B licence OK')"

In [ ]:
# 4. Offline tests (expected: 108 passed, 2 skipped)
!python -m pytest -q -p no:cacheprovider tests/ 2>&1 | tail -2

## Step 1 — S1b equivalence checks (bfloat16, one GPU)
All six checks must pass. Writes `results_9b/reports_m0/equivalence_gemma-2-9b-it.json`.

In [ ]:
t0 = time.time()
if not (REPO / 'reports/m0/equivalence_gemma-2-9b-it.json').exists():
    !bash scripts/s1_equivalence.sh gemma-2-9b-it cuda
    log('S1b equivalence', t0)
else:
    print('already done')
print(json.dumps({k: v['pass'] for k, v in json.load(open('reports/m0/equivalence_gemma-2-9b-it.json'))['checks'].items()}, indent=1))

## Step 2 — 1,000 GSM8K-train solutions (as pre-registered in `prereg/E0.md`)
Same 500 problems (seed 0) × marked and terse prompts, greedy, 512 new tokens. Resumes after a disconnect.

In [ ]:
t0 = time.time()
D = 'data/generated/gemma-2-9b-it'
!mkdir -p {D}
!python -m harness.datasets --source gsm8k --split train --n 500 --seed 0 --out data/m0_gsm8k_train500.jsonl
!python -c "from harness.generate import run; from harness.prompts import PromptSpec; run('gemma-2-9b-it', 'data/m0_gsm8k_train500.jsonl', '{D}/m0_traces.jsonl', specs=[PromptSpec('marked', 0), PromptSpec('terse', 0)])"
!python -m harness.score --traces {D}/m0_traces.jsonl --out {D}/m0_traces_scored.jsonl
n = sum(1 for _ in open(f'{D}/m0_traces.jsonl'))
log('generate 1000 traces', t0, traces=n)

In [ ]:
# speed: seconds and tokens per trace (for the S2 cost estimate)
import statistics as st
recs = [json.loads(l) for l in open(f'{D}/m0_traces_scored.jsonl')]
secs = [r['seconds'] for r in recs]; toks = [len(r['output_ids']) for r in recs]
summary = {"traces": len(recs), "accuracy": sum(r['correct'] for r in recs) / len(recs),
           "median_tokens": st.median(toks), "hit_max_tokens": sum(r['hit_max_tokens'] for r in recs),
           "median_seconds": st.median(secs), "total_hours": sum(secs) / 3600,
           "tokens_per_second": sum(toks) / sum(secs)}
print(json.dumps(summary, indent=1))
json.dump(summary, open(RES / 'reports_m0' / 'generation_speed_gemma-2-9b-it.json', 'w'), indent=1)

In [ ]:
# parser statistics on the 9B traces (no GPU needed)
t0 = time.time()
!python scripts/validate_parsing.py --traces {D}/m0_traces_scored.jsonl --out reports/m0/parsing_validation_gemma-2-9b-it.json --sample-out reports/m0/parsing_sample_gemma-2-9b-it.md --seed 5 > /dev/null
v = json.load(open('reports/m0/parsing_validation_gemma-2-9b-it.json'))['s1c_traces']
for k, x in v.items():
    print(k, 'parsed', x['traces_parsed'], 'has error step', round(x['trace_has_error_step'], 3), 's0 types', {a: round(b, 2) for a, b in x['s0_type_shares'].items()})
log('parser validation', t0)

## Step 3 — SAE validation, all 42 layers (A100 only)
Each 3.7 GB SAE is deleted after its layer. Finished layers are kept, so a disconnect loses at most one layer.

In [ ]:
assert BIG, 'Needs the A100.'
t0 = time.time()
os.environ['BTP_DROP_SAE_CACHE'] = '1'
!python -m analysis.sae_validation --model gemma-2-9b-it --traces {D}/m0_traces_scored.jsonl --max-traces 200 --out reports/m0/sae_validation_gemma-2-9b-it.json
r = json.load(open('reports/m0/sae_validation_gemma-2-9b-it.json'))
print('passing layers:', len(r['passing_layers']), 'of', len(r['layers']))
log('SAE validation', t0, passing=len(r['passing_layers']))

## Step 4 — E0 (A100 only)

In [ ]:
assert BIG, 'Needs the A100.'
t0 = time.time()
!python -m analysis.sparsity_e0 collect --model gemma-2-9b-it --traces {D}/m0_traces_scored.jsonl --sae-report reports/m0/sae_validation_gemma-2-9b-it.json --max-traces 1000 --out {D}/e0_tokens.csv.gz
!python -m analysis.sparsity_e0 analyse --tokens {D}/e0_tokens.csv.gz --n-layers 42 --out reports/m0/e0_gemma-2-9b-it.json > /dev/null
e = json.load(open('reports/m0/e0_gemma-2-9b-it.json'))
for b in ('early', 'middle', 'late'):
    if e.get(b):
        print(b, e[b]['layers'], 'dL0', round(e[b]['delta_L0']['estimate'], 2), 'd', round(e[b]['d_L0']['estimate'], 3), 'p_holm', e[b]['delta_L0']['p_holm'])
log('E0', t0)

## Done
Everything is in **My Drive / BTP / results_9b /**. Now **Runtime → Disconnect and delete runtime**.

In [ ]:
!pip freeze > {RES}/reports_m0/versions_colab.txt; nvidia-smi > {RES}/reports_m0/gpu_colab.txt
print(open(LOG).read())